# Phase 7 — Pretrain

Teach the model English before asking it to translate. Phase 4 failed because 474k words cannot teach language *and* a style mapping at once; this run supplies ~200M words.

**Kaggle settings — both required:**
- **Accelerator: GPU** (T4 or P100)
- **Internet: ON** — needed to clone the repo and stream Gutenberg

In [ ]:
import os, subprocess, sys, time

REPO = "https://github.com/Amay-M-Nair/AttentionMech.git"
ON_KAGGLE = os.path.isdir("/kaggle")
ROOT = "/kaggle/working/AttentionMech" if ON_KAGGLE else os.path.dirname(os.getcwd())

if ON_KAGGLE and not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--depth", "1", REPO, ROOT], check=True)

sys.path.insert(0, ROOT)

for pkg in ("sentencepiece", "datasets"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")
print("root:", ROOT)

## 1. Tokenise the corpus

Stream Gutenberg, tokenise once, write a flat `uint16` file. Doing it once means later epochs cost nothing and the dataloader never touches the network.

`MAX_WORDS` is the knob: 200M is the planned run, drop it to 20M for a quick end-to-end test.

In [ ]:
from src.corpus import stream_gutenberg, build_token_file, TokenFileDataset
from src.spm_tokenizer import SPMTokenizer

DATA = os.path.join(ROOT, "data")
TOKENS = "/kaggle/working/tokens.bin" if ON_KAGGLE else os.path.join(DATA, "tokens.bin")
MAX_WORDS = 200_000_000

tokenizer = SPMTokenizer(os.path.join(DATA, "spm32k.model"))
print("vocab:", len(tokenizer))

if not os.path.exists(TOKENS):
    t0 = time.time()
    n = build_token_file(stream_gutenberg(max_words=MAX_WORDS, progress_every=500),
                         tokenizer, TOKENS)
    print(f"{n/1e6:.1f}M tokens in {(time.time()-t0)/60:.1f} min")
else:
    print("token file already present")

print(f"{os.path.getsize(TOKENS)/1e6:.0f} MB")

Check the tokens decode back to English before training on them.

In [ ]:
SEQ_LEN = 256
tokens = TokenFileDataset(TOKENS, seq_len=SEQ_LEN)
print(f"{len(tokens):,} windows of {SEQ_LEN}")
print()
print(tokenizer.decode(tokens[0])[:300])

## 2. Span corruption

Mask 15% of tokens in spans of mean length 3. The encoder sees sentinels in place of the spans; the decoder reconstructs them. Filling those blanks is what teaches word meanings and syntax.

In [ ]:
from torch.utils.data import DataLoader
from src.dataset import collate_fn
from src.denoising import DenoisingDataset

BATCH = 32          # T4 16 GB; drop to 16 with ACCUMULATE 2 if OOM
ACCUMULATE = 1

dataset = DenoisingDataset(tokens, tokenizer.sentinel_ids)
loader = DataLoader(dataset, batch_size=BATCH, shuffle=True,
                    collate_fn=collate_fn, num_workers=2, drop_last=True)

src, tgt_in, tgt_out = next(iter(loader))
print("src    ", tuple(src.shape))
print("tgt_in ", tuple(tgt_in.shape))
print("tgt_out", tuple(tgt_out.shape))
print()
print("encoder:", tokenizer.decode(src[0], keep_specials=True)[:220])
print()
print("decoder:", tokenizer.decode(tgt_out[0], keep_specials=True)[:220])

## 3. The model

Same architecture as Phase 1 — scaling is a config change, nothing in `transformer.py` moves.

In [ ]:
from src.config import TransformerConfig, get_device
from src.transformer import build_model

device = get_device()
torch.manual_seed(0)

config = TransformerConfig(vocab_size=len(tokenizer), d_model=384, num_heads=8,
                           num_layers=6, d_ff=1536, dropout=0.1, max_len=512)
model = build_model(config).to(device)

for name, count in model.count_parameters().items():
    print(f"  {name:<12} {count:>12,}")

## 4. Gate — can it learn the objective?

Overfit a single batch. If it can't memorise one batch the corruption is wired wrong, and a full run would take an hour to tell you that.

A 37M model needs ~600 steps to get there. Expect roughly **loss 10.4 → 0.7, accuracy ~93%** — measured, not a guess.

In [ ]:
from src.train import overfit_batch

probe = build_model(TransformerConfig(vocab_size=len(tokenizer), d_model=384, num_heads=8,
                                      num_layers=6, d_ff=1536, dropout=0.0, max_len=512)).to(device)
batch = tuple(t.to(device) for t in next(iter(loader)))

h = overfit_batch(probe, batch, steps=600, log_every=150)
print(f"\nloss {h['loss'][0]:.3f} -> {h['loss'][-1]:.4f}   acc {h['acc'][-1]:.1%}")
passed = h["loss"][-1] < 1.5 and h["acc"][-1] > 0.85
print("GATE:", "PASS" if passed else "FAIL - do not start the long run")
del probe
torch.cuda.empty_cache()

## 5. Measure throughput before committing

Time estimates carry large error bars. 300 steps gives a real tokens/sec, and the full run length follows from it.

In [ ]:
from src.train import pretrain

PROBE_CK = "/kaggle/working/_probe.pt" if ON_KAGGLE else os.path.join(ROOT, "checkpoints", "_probe.pt")
if os.path.exists(PROBE_CK):
    os.remove(PROBE_CK)

probe_history = pretrain(model, config, loader, device, steps=300, lr=3e-4, warmup=100,
                         accumulate=ACCUMULATE, log_every=100,
                         checkpoint_path=PROBE_CK, checkpoint_every=0, resume=False)

rate = probe_history["tokens_per_sec"][-1]
tokens_total = len(tokens) * SEQ_LEN
steps_per_epoch = tokens_total // (BATCH * ACCUMULATE * SEQ_LEN)

print(f"\n{rate/1000:.1f}k corpus tokens/sec")
print(f"corpus: {tokens_total/1e6:.0f}M tokens -> {steps_per_epoch:,} steps for one pass")
print(f"one pass: {tokens_total/rate/3600:.1f} h")
os.remove(PROBE_CK)

## 6. The run

Set `STEPS` from the estimate above — one pass over the corpus is the target. More data at one epoch beats repeating less data.

Checkpoints carry optimizer and scheduler state, so a dead session resumes correctly. Re-running this cell picks up where it stopped.

In [ ]:
STEPS = steps_per_epoch
CHECKPOINT = "/kaggle/working/pretrained.pt" if ON_KAGGLE else os.path.join(ROOT, "checkpoints", "pretrained.pt")

torch.manual_seed(0)
model = build_model(config).to(device)

history = pretrain(
    model, config, loader, device,
    steps=STEPS, lr=3e-4, warmup=2000,
    accumulate=ACCUMULATE, amp=True, log_every=200,
    checkpoint_path=CHECKPOINT, checkpoint_every=1000, resume=True,
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["step"], history["loss"])
axes[0].set_xlabel("step"); axes[0].set_ylabel("loss"); axes[0].grid(alpha=0.3)
axes[0].set_title("Training loss")

axes[1].plot(history["step"], history["perplexity"], color="tab:orange")
axes[1].set_yscale("log")
axes[1].set_xlabel("step"); axes[1].set_ylabel("perplexity (log)"); axes[1].grid(alpha=0.3)
axes[1].set_title("Perplexity")
plt.tight_layout(); plt.show()

print(f"final loss {history['loss'][-1]:.3f}   perplexity {history['perplexity'][-1]:.1f}")
print(f"random baseline would be {len(tokenizer):,}")

## 7. Gate — did it learn English?

Perplexity falling is necessary but not sufficient. Mask a span in a sentence the model has never seen and read what it fills in.

In [ ]:
from src.inference import greedy_decode
from src.dataset import load_split
from src.denoising import corrupt_spans
import numpy as np

held_out, _ = load_split(DATA, "test")
rng = np.random.default_rng(0)
model.eval()

for line in held_out[:6]:
    ids = tokenizer.encode(line)
    if len(ids) < 12:
        continue
    enc, _ = corrupt_spans(ids, tokenizer.sentinel_ids, rng=rng)
    src_t = torch.tensor([enc + [2]], device=device)
    out = greedy_decode(model, src_t, max_len=48)

    print("masked :", tokenizer.decode(enc, keep_specials=True))
    print("filled :", tokenizer.decode(out[0], keep_specials=True))
    print("truth  :", line)
    print()

## Done

`pretrained.pt` now holds a model that knows English. Download it from the Kaggle output panel — Phase 8 loads it with `load_pretrained()` and fine-tunes on the 18k Shakespeare pairs.

The target there: **beat 19.22**, the copy baseline that Phase 4 scored 15.07 against.